Task 08 | Cache Decorator with Expiration

In [1]:
import inspect
import time
from functools import wraps


def ttl_cache(seconds):
    if seconds < 0:
        raise ValueError("seconds cannot be negative")

    def decorator(func):
        cache = {}
        hits = 0
        misses = 0
        signature = inspect.signature(func)

        @wraps(func)
        def wrapper(*args, **kwargs):
            nonlocal hits, misses

            bound = signature.bind(*args, **kwargs)
            bound.apply_defaults()

            key_parts = []
            for name, parameter in signature.parameters.items():
                value = bound.arguments[name]

                if parameter.kind == inspect.Parameter.VAR_KEYWORD:
                    value = tuple(sorted(value.items()))
                elif parameter.kind == inspect.Parameter.VAR_POSITIONAL:
                    value = tuple(value)

                key_parts.append((name, value))

            key = tuple(key_parts)

            try:
                hash(key)
            except TypeError:
                misses += 1
                return func(*args, **kwargs)

            now = time.monotonic()

            if key in cache:
                expires_at, result = cache[key]
                if now < expires_at:
                    hits += 1
                    return result
                del cache[key]

            misses += 1
            result = func(*args, **kwargs)
            cache[key] = (now + seconds, result)
            return result

        def cache_info():
            return {"hits": hits, "misses": misses, "size": len(cache)}

        wrapper.cache_info = cache_info
        return wrapper

    return decorator


@ttl_cache(10)
def add(x, y):
    print("Calculating...")
    return x + y


print(add(x=2, y=3))
print(add(y=3, x=2))  # Cache hit
print(add.cache_info())

Calculating...
5
5
{'hits': 1, 'misses': 1, 'size': 1}
